Giải Sudoku 4×4 hoặc 6×6​

Biểu diễn bài toán như tìm kiếm trạng thái. Áp dụng DFS với backtracking, so sánh với chiến lược chọn ô có ít khả năng nhất (MRV heuristic).​

# Solve Sudoku 4x4 and 6x6 

## What is sudoku?

- Sudoku is a popular logic-based number-placement puzzle.

- To complete the puzzle, you must fill all the empty cells so that every row, every column, and every nxn box contains the numbers 1 through n square without repeating any number.

#### Basic rule

- No Repeating Numbers: Each number from 1 to n square can appear only once in each row, column, and nxn box.

- Logic Over Math: Sudoku requires no mathematical calculations or arithmetic skills—it relies purely on pattern recognition and logical deduction.

- Single Solution: A well-designed Sudoku puzzle has only one unique solution.

## What algorithm should we implement?

- To solve this problem, we use Depth-First Search and Backtracking
- Because:
    - Guarantees Finding an Exact Solution (Completeness)
    - Space Efficiency ($O(D)$ Memory)


## Algorithm Implement

### Sodoku patterns

In [880]:
import time

In [881]:
sodoku_four = [
    [0, 0, 4, 0],
    [2, 0, 0, 0],
    [0, 0, 0, 4],
    [0, 3, 0, 0],
]      #Sodoku 4x4

sodoku_six = [
    [0, 0, 0, 0, 0, 0], 
    [6, 0, 2, 5, 0, 0], 
    [3, 0, 0, 2, 5, 0], 
    [0, 2, 5, 0, 0, 4], 
    [0, 0, 3, 1, 0, 5], 
    [0, 0, 0, 0, 0, 0], 
]

- Check validation of sodoku board

In [882]:
def valid(sodoku_board):
    size = len(sodoku_board)
    
    def valid_line(arr): #Satisfying condition
        new_arr = [i for i in arr if i != 0] # New array only contains the number filled.
        for num in new_arr:
            if arr.count(num) != 1: #Counting the number of appearance of each element in new_arr
                return False # If it appears more than once -> violating rule -> return False
        return True # else return True
    
    def valid_in_row(sodoku_board): #Row
        for r in range(size):
            if not valid_line(sodoku_board[r]):
                return False
        return True
        
    def valid_in_col(sodoku_board): #Col
        for r in range(size):
            col = []
            for c in range(size):
                col.append(sodoku_board[c][r])
        if not valid_line(col):
            return False
        return True
    
    def valid_in_box(sodoku_board): #Box
        if size == 4:
            box_r, box_c = 2, 2
        elif size == 6:
            box_r, box_c = 2, 3

        for i in range(0, size, box_r):
            for j in range(0, size, box_c):
                box = []
                for r in range(i, i + box_r):
                    for c in range(j, j + box_c):
                        box.append(sodoku_board[r][c])
                if not valid_line(box): # Gom đủ 1 box rồi mới kiểm tra
                    return False
        return True

    return valid_in_row(sodoku_board) and valid_in_col(sodoku_board) and valid_in_box(sodoku_board)

- Check valid of sodoku board as soon as filling a number

In [883]:
def isvalid(board, row, column, num):
    size = len(board)

    for i in range(len(board)):
        if board[row][i] == num: #Iterating all number in row if appears return False else return True
            return False
        
        if board[i][column] == num: #Col
            return False
    
    if size == 4:
        start_r = (row // 2) * 2 # row position of cell at corner
        start_c = (column // 2) * 2 # column position of cell at corner
            
        
        for r in range(start_r, start_r + (size // 2)):
            for c in range(start_c, start_c + (size // 2)):
                if   board[r][c] == num:
                    return False
    if size == 6:
        start_r = (row // 2) * 2
        start_c = (column // 3) * 3
        
        for r in range(start_r, start_r + 2):
            for c in range(start_c, start_c + 3):
                if board[r][c] == num:
                    return False
    return True
                

- Solve sodoku

In [884]:
def solveSodoku(board):
    node_count = 0
    backtrack_count = 0
    start_time = time.perf_counter()
    def solve():
        nonlocal node_count, backtrack_count, start_time
        size = len(board)
        for r in range(size):
            for c in range(size):
                if board[r][c] == 0: # Check if cell is filled
                    node_count += 1
                    for opt in range(1, size + 1): # Try number from  1 to n
                        if isvalid(board, r, c, opt): # Check validation if fill number to this cell
                            board[r][c] = opt #Fill
                            
                            if solve(): #continue descending deeper
                                return True #If the board was solved, return True
                            
                            board[r][c] = 0 # If the board is not solved, backtracking
                            backtrack_count += 1
                    return False # If try all number from 1 to n and board still is not solved, return False to backtracking
        return True
    if solve():
        execution_time = (time.perf_counter() - start_time) * 1000
        return board ,node_count, backtrack_count, execution_time
    return None, node_count, backtrack_count, 0


## MRV strategy

#### What is MRV strategy?

- As we can see, solveSodoku() go through each cell, try all cases from 1 to size + 1 regardless of whether 1 or 5 possible values -> many steps backtrack.
- On the other hand, MRV help we select the cell with the fewest remaining valid values. -> decrease step backtrack.

#### New pattern


In [885]:
new_sodoku_four = [
    [0, 0, 4, 0],
    [2, 0, 0, 0],
    [0, 0, 0, 4],
    [0, 3, 0, 0],
]      #Sodoku 4x4

new_sodoku_six = [
    [0, 0, 0, 0, 0, 0], 
    [6, 0, 2, 5, 0, 0], 
    [3, 0, 0, 2, 5, 0], 
    [0, 2, 5, 0, 0, 4], 
    [0, 0, 3, 1, 0, 5], 
    [0, 0, 0, 0, 0, 0], 
]

In [886]:
def find_best_cell(board):
    
    #Init all essential variable
    min_option = 100000000000000
    best_cell = None
    size = len(board)
    
    for r in range(size):
        for c in range(size): 
            if board[r][c] == 0: # Check if cell is filled
                valid_count = 0 # The number of validating numbers
                for num in range(1, size + 1):
                    if isvalid(board, r, c, num):
                        valid_count += 1
                if valid_count < min_option:
                    min_option = valid_count
                    best_cell = (r, c) # The cell with min validating numbers
                    
    return best_cell

In [887]:
def solveSodoku_with_MRV(board):
    node_count = 0
    backtrack_count = 0
    start_time = time.perf_counter() 
    def solve():
        nonlocal node_count, backtrack_count, start_time
        cell = find_best_cell(board)
        
        if not cell:
            return True
        r, c = cell
        size = len(board)
        node_count += 1
        
        for num in range(1, size + 1):
            if isvalid(board, r, c, num):
                board[r][c] = num

                if solve():
                    return True
                board[r][c] = 0
                backtrack_count += 1
                
        return False
    if solve():
        end_time = time.perf_counter()
        execution_time = (end_time - start_time) * 1000  # Đổi sang mili-giây (ms)
        return board, node_count, backtrack_count, execution_time
    return None, node_count, backtrack_count, 0

## Execution time and comparision

In [888]:
mrv_sodoku_four = [
    [0, 0, 4, 0],
    [2, 0, 0, 0],
    [0, 0, 0, 4],
    [0, 3, 0, 0],
]

mrv_sodoku_six = [
    [0, 0, 0, 0, 0, 0], 
    [6, 0, 2, 5, 0, 0], 
    [3, 0, 0, 2, 5, 0], 
    [0, 2, 5, 0, 0, 4], 
    [0, 0, 3, 1, 0, 5], 
    [0, 0, 0, 0, 0, 0], 
]

# Solve 2 test case with MRV
print("=== SUDOKU 4x4 ===")
solved_board_4, nodes_4, bt_4, time_4 = solveSodoku(mrv_sodoku_four)
if valid(solved_board_4):
    print(f"Nodes: {nodes_4}")
    print(f"Backtrack: {bt_4}")
    print(f"Execution Time: {time_4:.4f} ms")
    for row in solved_board_4:
        print(row)
else:
    print("Incorrect solution")

print("\n=== SUDOKU 6x6 ===")
solved_board_6, nodes_6, bt_6, time_6 = solveSodoku(mrv_sodoku_six)
if valid(solved_board_6):
    print(f"Nodes: {nodes_6}")
    print(f"Backtrack: {bt_6}")
    print(f"Execution Time: {time_6:.4f} ms")
    for row in solved_board_6:
        print(row)
else:
    print("Incorrect solution")
    


=== SUDOKU 4x4 ===
Nodes: 13
Backtrack: 1
Execution Time: 0.0705 ms
[3, 1, 4, 2]
[2, 4, 1, 3]
[1, 2, 3, 4]
[4, 3, 2, 1]

=== SUDOKU 6x6 ===
Nodes: 292
Backtrack: 268
Execution Time: 1.5992 ms
[5, 3, 4, 6, 1, 2]
[6, 1, 2, 5, 4, 3]
[3, 4, 6, 2, 5, 1]
[1, 2, 5, 3, 6, 4]
[4, 6, 3, 1, 2, 5]
[2, 5, 1, 4, 3, 6]


In [889]:
# Khởi tạo lại bảng mẫu để chạy thuật toán MRV
mrv_sodoku_four = [
    [0, 0, 4, 0],
    [2, 0, 0, 0],
    [0, 0, 0, 4],
    [0, 3, 0, 0],
]

mrv_sodoku_six = [
    [0, 0, 0, 0, 0, 0], 
    [6, 0, 2, 5, 0, 0], 
    [3, 0, 0, 2, 5, 0], 
    [0, 2, 5, 0, 0, 4], 
    [0, 0, 3, 1, 0, 5], 
    [0, 0, 0, 0, 0, 0], 
]

# Solve with DFS
print("=== SUDOKU 4x4 (VỚI MRV) ===")
solved_board_4, nodes_4, bt_4, time_4 = solveSodoku_with_MRV(mrv_sodoku_four)
if valid(solved_board_4):
    print(f"Nodes: {nodes_4}")
    print(f"Backtrack: {bt_4}")
    print(f"Execution Time: {time_4:.4f} ms")
    for row in solved_board_4:
        print(row)
else:
    print("Incorrect solution")

print("\n=== SUDOKU 6x6 (VỚI MRV) ===")
solved_board_6, nodes_6, bt_6, time_6 = solveSodoku_with_MRV(mrv_sodoku_six)
if valid(solved_board_6):
    print(f"Nodes: {nodes_6}")
    print(f"Backtrack: {bt_6}")
    print(f"Execution Time: {time_6:.4f} ms")
    for row in solved_board_6:
        print(row)
else:
    print("Incorrect solution") 
    

=== SUDOKU 4x4 (VỚI MRV) ===
Nodes: 12
Backtrack: 0
Execution Time: 0.3411 ms
[3, 1, 4, 2]
[2, 4, 1, 3]
[1, 2, 3, 4]
[4, 3, 2, 1]

=== SUDOKU 6x6 (VỚI MRV) ===
Nodes: 41
Backtrack: 17
Execution Time: 5.6103 ms
[5, 3, 4, 6, 1, 2]
[6, 1, 2, 5, 4, 3]
[3, 4, 6, 2, 5, 1]
[1, 2, 5, 3, 6, 4]
[4, 6, 3, 1, 2, 5]
[2, 5, 1, 4, 3, 6]
